# 🔍 Intelligent General-Purpose EDA Notebook
## Comprehensive Data Exploration & Analysis Framework

**Purpose:** Automatic, intelligent analysis of any dataset

- ✅ Works with ANY data type (CSV, JSON, Excel, SQL, etc.)
- ✅ Object-Oriented Design for modularity & reusability
- ✅ Comprehensive statistical & visual analysis
- ✅ Automatic outlier detection & analysis
- ✅ Correlation analysis & relationships
- ✅ Deep insights & actionable recommendations

## 1️⃣ IMPORTS & CONFIGURATION

In [ ]:
# Data Handling
import pandas as pd
import numpy as np
from pathlib import Path
import json

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Statistical Analysis
from scipy import stats
from scipy.stats import skew, kurtosis, chi2_contingency
from sklearn.preprocessing import StandardScaler

# Utilities
import datetime
from typing import Dict, List, Tuple, Any
from dataclasses import dataclass
from enum import Enum

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

print('✅ All libraries imported successfully')

## 2️⃣ DEFINE ANALYSIS CLASSES (OOP Architecture)

In [ ]:
class DataQuality:
    """Assess data quality and integrity"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.report = {}
    
    def analyze(self) -> Dict[str, Any]:
        self.report = {
            'shape': self.df.shape,
            'memory_usage': f"{self.df.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
            'missing_values': self._missing_analysis(),
            'duplicates': self._duplicate_analysis(),
            'data_types': self._dtype_analysis(),
            'completeness': self._completeness_score()
        }
        return self.report
    
    def _missing_analysis(self) -> Dict:
        missing = self.df.isnull().sum()
        missing_pct = (missing / len(self.df)) * 100
        return missing[missing > 0].to_dict() if missing.sum() > 0 else {}
    
    def _duplicate_analysis(self) -> Dict:
        total_dupes = self.df.duplicated().sum()
        return {
            'total_duplicates': total_dupes,
            'percentage': (total_dupes / len(self.df)) * 100
        }
    
    def _dtype_analysis(self) -> Dict:
        return self.df.dtypes.value_counts().to_dict()
    
    def _completeness_score(self) -> float:
        return (1 - self.df.isnull().sum().sum() / (len(self.df) * len(self.df.columns))) * 100
    
    def display(self):
        print('\n' + '='*100)
        print('📊 DATA QUALITY REPORT')
        print('='*100)
        print(f"\n📐 Shape: {self.report['shape'][0]:,} rows × {self.report['shape'][1]} columns")
        print(f"💾 Memory Usage: {self.report['memory_usage']}")
        print(f"✅ Completeness Score: {self.report['completeness']:.2f}%")
        
        if self.report['missing_values']:
            print(f"\n⚠️  Missing Values:")
            for col, count in self.report['missing_values'].items():
                pct = (count / len(self.df)) * 100
                print(f"  • {col}: {count} ({pct:.2f}%)")
        else:
            print(f"\n✅ No missing values found!")
        
        print(f"\n🔄 Duplicates: {self.report['duplicates']['total_duplicates']} ({self.report['duplicates']['percentage']:.2f}%)")
        print(f"\n📝 Data Types:")
        for dtype, count in self.report['data_types'].items():
            print(f"  • {dtype}: {count}")

In [ ]:
class NumericAnalyzer:
    """Comprehensive numeric column analysis"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    def analyze(self) -> Dict[str, Any]:
        analysis = {}
        for col in self.numeric_cols:
            analysis[col] = self._analyze_column(col)
        return analysis
    
    def _analyze_column(self, col: str) -> Dict[str, Any]:
        data = self.df[col].dropna()
        
        return {
            'count': len(data),
            'mean': float(data.mean()),
            'median': float(data.median()),
            'std': float(data.std()),
            'min': float(data.min()),
            'q25': float(data.quantile(0.25)),
            'q75': float(data.quantile(0.75)),
            'max': float(data.max()),
            'iqr': float(data.quantile(0.75) - data.quantile(0.25)),
            'skewness': float(skew(data)),
            'kurtosis': float(kurtosis(data)),
            'variance': float(data.var()),
            'cv': float(data.std() / abs(data.mean()) if data.mean() != 0 else 0)
        }
    
    def visualize(self):
        if not self.numeric_cols:
            print("No numeric columns found")
            return
        
        n_cols = len(self.numeric_cols)
        n_rows = (n_cols + 2) // 3
        
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5*n_rows))
        axes = axes.flatten() if n_cols > 1 else [axes]
        
        for idx, col in enumerate(self.numeric_cols):
            axes[idx].hist(self.df[col].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
            axes[idx].set_title(f'{col}\n(Skew: {skew(self.df[col].dropna()):.2f}, Kurt: {kurtosis(self.df[col].dropna()):.2f})', 
                              fontweight='bold')
            axes[idx].set_xlabel('Value')
            axes[idx].set_ylabel('Frequency')
            axes[idx].grid(axis='y', alpha=0.3)
        
        for idx in range(len(self.numeric_cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle('📊 Numeric Columns Distribution', fontsize=16, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.show()
    
    def display_stats(self):
        print('\n' + '='*100)
        print('📈 NUMERIC ANALYSIS')
        print('='*100)
        
        for col, stats_dict in self.analyze().items():
            print(f"\n🔢 {col}")
            print(f"  Count: {stats_dict['count']:.0f} | Mean: {stats_dict['mean']:.4f} | Median: {stats_dict['median']:.4f}")
            print(f"  Std: {stats_dict['std']:.4f} | Min: {stats_dict['min']:.4f} | Max: {stats_dict['max']:.4f}")
            print(f"  Q1: {stats_dict['q25']:.4f} | Q3: {stats_dict['q75']:.4f} | IQR: {stats_dict['iqr']:.4f}")
            print(f"  Skewness: {stats_dict['skewness']:.4f} | Kurtosis: {stats_dict['kurtosis']:.4f} | CV: {stats_dict['cv']:.4f}")

In [ ]:
class CategoricalAnalyzer:
    """Comprehensive categorical column analysis"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    def analyze(self) -> Dict[str, Any]:
        analysis = {}
        for col in self.categorical_cols:
            vc = self.df[col].value_counts()
            analysis[col] = {
                'unique_count': self.df[col].nunique(),
                'missing': int(self.df[col].isnull().sum()),
                'top_value': str(vc.index[0]) if len(vc) > 0 else None,
                'top_frequency': int(vc.iloc[0]) if len(vc) > 0 else 0,
                'cardinality': float(self.df[col].nunique() / len(self.df)),
                'value_counts': vc.to_dict()
            }
        return analysis
    
    def visualize(self):
        if not self.categorical_cols:
            print("No categorical columns found")
            return
        
        n_cols = len(self.categorical_cols)
        n_rows = (n_cols + 2) // 3
        
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5*n_rows))
        axes = axes.flatten() if n_cols > 1 else [axes]
        
        for idx, col in enumerate(self.categorical_cols):
            top_n = min(10, self.df[col].nunique())
            top_values = self.df[col].value_counts().head(top_n)
            
            top_values.plot(kind='barh', ax=axes[idx], color='teal')
            axes[idx].set_title(f'{col}\n(Unique: {self.df[col].nunique()})', fontweight='bold')
            axes[idx].set_xlabel('Frequency')
        
        for idx in range(len(self.categorical_cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle('📊 Categorical Columns Distribution (Top 10)', fontsize=16, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.show()
    
    def display_stats(self):
        print('\n' + '='*100)
        print('📋 CATEGORICAL ANALYSIS')
        print('='*100)
        
        for col, col_stats in self.analyze().items():
            print(f"\n🏷️  {col}")
            print(f"  Unique Values: {col_stats['unique_count']} | Missing: {col_stats['missing']} | Cardinality: {col_stats['cardinality']:.4f}")
            print(f"  Most Frequent: {col_stats['top_value']} ({col_stats['top_frequency']} occurrences)")
            if col_stats['unique_count'] <= 10:
                print(f"  Value Distribution:")
                for val, freq in list(col_stats['value_counts'].items())[:5]:
                    pct = (freq / len(self.df)) * 100
                    print(f"    • {val}: {freq} ({pct:.2f}%)")

In [ ]:
class OutlierDetection:
    """Detect and analyze outliers"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    def detect_iqr(self) -> Dict[str, Dict]:
        """IQR-based outlier detection"""
        outliers = {}
        for col in self.numeric_cols:
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_mask = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
            if outlier_mask.sum() > 0:
                outliers[col] = {
                    'count': int(outlier_mask.sum()),
                    'percentage': float((outlier_mask.sum() / len(self.df)) * 100),
                    'lower_bound': float(lower_bound),
                    'upper_bound': float(upper_bound)
                }
        return outliers
    
    def detect_zscore(self, threshold=3) -> Dict[str, int]:
        """Z-score based outlier detection"""
        outliers = {}
        for col in self.numeric_cols:
            z_scores = np.abs(stats.zscore(self.df[col].dropna()))
            outlier_count = (z_scores > threshold).sum()
            if outlier_count > 0:
                outliers[col] = int(outlier_count)
        return outliers
    
    def visualize(self):
        if not self.numeric_cols:
            return
        
        n_cols = len(self.numeric_cols)
        n_rows = (n_cols + 2) // 3
        
        fig, axes = plt.subplots(n_rows, 3, figsize=(18, 5*n_rows))
        axes = axes.flatten() if n_cols > 1 else [axes]
        
        for idx, col in enumerate(self.numeric_cols):
            axes[idx].boxplot(self.df[col].dropna())
            axes[idx].set_title(f'{col}', fontweight='bold')
            axes[idx].set_ylabel('Value')
            axes[idx].grid(axis='y', alpha=0.3)
        
        for idx in range(len(self.numeric_cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle('📦 Outlier Detection (Box Plots)', fontsize=16, fontweight='bold', y=1.00)
        plt.tight_layout()
        plt.show()
    
    def display(self):
        outliers_iqr = self.detect_iqr()
        outliers_zscore = self.detect_zscore()
        
        print('\n' + '='*100)
        print('🎯 OUTLIER DETECTION')
        print('='*100)
        
        if outliers_iqr:
            print('\n📌 IQR Method:')
            for col, info in outliers_iqr.items():
                print(f"  • {col}: {info['count']} outliers ({info['percentage']:.2f}%)")
                print(f"    Bounds: [{info['lower_bound']:.4f}, {info['upper_bound']:.4f}]")
        else:
            print('\n✅ No outliers detected (IQR method)')
        
        if outliers_zscore:
            print('\n📌 Z-Score Method (threshold=3):')
            for col, count in outliers_zscore.items():
                print(f"  • {col}: {count} outliers")
        else:
            print('\n✅ No outliers detected (Z-Score method)')

In [ ]:
class CorrelationAnalyzer:
    """Analyze correlations between variables"""
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    def compute_correlation(self) -> pd.DataFrame:
        if len(self.numeric_cols) < 2:
            return None
        return self.df[self.numeric_cols].corr()
    
    def get_high_correlations(self, threshold=0.7) -> List[Tuple]:
        """Find highly correlated feature pairs"""
        corr_matrix = self.compute_correlation()
        if corr_matrix is None:
            return []
        
        high_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) >= threshold:
                    high_corr.append((
                        str(corr_matrix.columns[i]),
                        str(corr_matrix.columns[j]),
                        float(corr_matrix.iloc[i, j])
                    ))
        return sorted(high_corr, key=lambda x: abs(x[2]), reverse=True)
    
    def visualize(self):
        corr_matrix = self.compute_correlation()
        if corr_matrix is None:
            print("Need at least 2 numeric columns")
            return
        
        plt.figure(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                   square=True, linewidths=1, cbar_kws={"shrink": 0.8})
        plt.title('📊 Correlation Matrix', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    def display(self):
        high_corr = self.get_high_correlations()
        
        print('\n' + '='*100)
        print('🔗 CORRELATION ANALYSIS')
        print('='*100)
        
        if high_corr:
            print(f'\n🔴 Highly Correlated Pairs (|r| ≥ 0.7):')
            for col1, col2, corr_val in high_corr:
                print(f"  • {col1} ↔ {col2}: {corr_val:.4f}")
        else:
            print(f'\n✅ No highly correlated pairs (|r| ≥ 0.7) found')

In [ ]:
class DataProfiler:
    """Unified data profiler coordinating all analyses"""
    
    def __init__(self, df: pd.DataFrame, name: str = "Dataset"):
        self.df = df
        self.name = name
        
        # Initialize all analyzers
        self.quality = DataQuality(df)
        self.numeric = NumericAnalyzer(df)
        self.categorical = CategoricalAnalyzer(df)
        self.outliers = OutlierDetection(df)
        self.correlation = CorrelationAnalyzer(df)
    
    def run_full_analysis(self):
        """Execute comprehensive analysis"""
        print(f"\n\n{'='*100}")
        print(f'🎯 COMPREHENSIVE DATA PROFILING: {self.name.upper()}')
        print(f"{'='*100}")
        print(f"Analysis started: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        # Step 1: Data Quality
        self.quality.analyze()
        self.quality.display()
        
        # Step 2: Numeric Analysis
        if self.numeric.numeric_cols:
            self.numeric.display_stats()
            print("\n📊 Generating numeric distributions...")
            self.numeric.visualize()
        
        # Step 3: Categorical Analysis
        if self.categorical.categorical_cols:
            self.categorical.display_stats()
            print("\n📊 Generating categorical distributions...")
            self.categorical.visualize()
        
        # Step 4: Outlier Detection
        if self.numeric.numeric_cols:
            self.outliers.display()
            print("\n📊 Generating box plots...")
            self.outliers.visualize()
        
        # Step 5: Correlation Analysis
        if len(self.numeric.numeric_cols) >= 2:
            self.correlation.display()
            print("\n📊 Generating correlation heatmap...")
            self.correlation.visualize()
        
        # Final Summary
        self._print_summary()
    
    def _print_summary(self):
        print(f"\n\n{'='*100}")
        print('📋 ANALYSIS SUMMARY')
        print(f"{'='*100}")
        print(f"\n✅ Total Columns: {len(self.df.columns)}")
        print(f"   • Numeric: {len(self.numeric.numeric_cols)}")
        print(f"   • Categorical: {len(self.categorical.categorical_cols)}")
        print(f"\n📊 Data Quality:")
        print(f"   • Completeness: {self.quality.report['completeness']:.2f}%")
        print(f"   • Duplicates: {self.quality.report['duplicates']['total_duplicates']}")
        print(f"\n🎯 Next Steps:")
        print(f"   1. Handle missing values (fill/drop)")
        print(f"   2. Treat outliers (if needed)")
        print(f"   3. Feature engineering & scaling")
        print(f"   4. Prepare for modeling")
        print(f"\n" + '='*100 + "\n")

## 3️⃣ LOAD YOUR DATASET

In [ ]:
# ============================================
# 📂 CONFIGURE YOUR DATA SOURCE
# ============================================

# Option 1: Load from CSV
df = pd.read_csv('your_data.csv')

# Option 2: Load from Excel
# df = pd.read_excel('your_data.xlsx')

# Option 3: Load from JSON
# df = pd.read_json('your_data.json')

# Option 4: Sample data for testing
# Uncomment to use sample data:
# df = pd.DataFrame({
#     'age': np.random.randint(18, 80, 1000),
#     'salary': np.random.randint(30000, 200000, 1000),
#     'department': np.random.choice(['HR', 'IT', 'Finance', 'Sales'], 1000),
#     'experience_years': np.random.randint(0, 30, 1000),
#     'years_at_company': np.random.randint(0, 20, 1000)
# })

print(f"✅ Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

## 4️⃣ RUN COMPREHENSIVE ANALYSIS

In [ ]:
# Initialize the profiler with your dataset name
profiler = DataProfiler(df, name="Your Dataset Name")

# Run the complete analysis
profiler.run_full_analysis()

## 5️⃣ ADVANCED INSIGHTS & CUSTOM ANALYSIS

In [ ]:
# ============================================
# 🔬 CUSTOM ANALYSIS SECTION
# ============================================

# Example 1: Focus on a specific numeric column
if profiler.numeric.numeric_cols:
    col = profiler.numeric.numeric_cols[0]  # Change index as needed
    print(f'\n\n📊 Detailed Analysis of: {col}')
    print('='*80)
    
    data = df[col].dropna()
    print(f'\n📈 Statistical Summary:')
    print(f'  Mean: {data.mean():.4f}')
    print(f'  Median: {data.median():.4f}')
    print(f'  Mode: {data.mode().values[0] if len(data.mode()) > 0 else "N/A"}')
    print(f'  Range: [{data.min():.4f}, {data.max():.4f}]')
    print(f'  Standard Deviation: {data.std():.4f}')
    print(f'  Variance: {data.var():.4f}')
    print(f'\n🎯 Distribution Shape:')
    print(f'  Skewness: {skew(data):.4f}')
    if skew(data) > 0.5:
        print(f'    → Right-skewed distribution (tail on right)')
    elif skew(data) < -0.5:
        print(f'    → Left-skewed distribution (tail on left)')
    else:
        print(f'    → Approximately symmetric distribution')
    print(f'  Kurtosis: {kurtosis(data):.4f}')
    print(f'    → Peak sharpness measure')

In [ ]:
# Example 2: Analyze relationships between numeric columns
if len(profiler.numeric.numeric_cols) >= 2:
    col1, col2 = profiler.numeric.numeric_cols[0], profiler.numeric.numeric_cols[1]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Scatter plot
    axes[0].scatter(df[col1], df[col2], alpha=0.6, s=30)
    axes[0].set_xlabel(col1, fontsize=12)
    axes[0].set_ylabel(col2, fontsize=12)
    axes[0].set_title(f'Relationship: {col1} vs {col2}', fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Correlation
    corr = df[[col1, col2]].corr().iloc[0, 1]
    axes[1].text(0.5, 0.5, f'Correlation\n{corr:.4f}', 
                ha='center', va='center', fontsize=20, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].axis('off')
    
    plt.suptitle('📊 Pairwise Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6️⃣ DATA PREPROCESSING TEMPLATE

In [ ]:
# ============================================
# 🔧 DATA PREPROCESSING PIPELINE
# ============================================

# Step 1: Handle Missing Values
def handle_missing_values(df, strategy='mean'):
    """Handle missing values smartly"""
    df_clean = df.copy()
    
    # Numeric columns
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            if strategy == 'mean':
                df_clean[col].fillna(df_clean[col].mean(), inplace=True)
            elif strategy == 'median':
                df_clean[col].fillna(df_clean[col].median(), inplace=True)
            elif strategy == 'drop':
                df_clean.dropna(subset=[col], inplace=True)
    
    # Categorical columns
    cat_cols = df_clean.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    
    return df_clean

# Apply missing value handling
# df_clean = handle_missing_values(df, strategy='mean')
# print(f"✅ Missing values handled")
# print(f"Missing values after: {df_clean.isnull().sum().sum()}")

In [ ]:
# Step 2: Remove Duplicates
def remove_duplicates(df):
    """Remove duplicate rows"""
    initial_shape = df.shape
    df_clean = df.drop_duplicates().reset_index(drop=True)
    removed = initial_shape[0] - df_clean.shape[0]
    print(f"✅ Removed {removed} duplicate rows")
    return df_clean

# df_clean = remove_duplicates(df)

In [ ]:
# Step 3: Outlier Treatment
def handle_outliers(df, method='iqr'):
    """Handle outliers using IQR method"""
    df_clean = df.copy()
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if method == 'iqr':
            Q1 = df_clean[col].quantile(0.25)
            Q3 = df_clean[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            df_clean[col] = df_clean[col].clip(lower, upper)
    
    return df_clean

# df_clean = handle_outliers(df)
# print("✅ Outliers handled using IQR method")

In [ ]:
# Step 4: Feature Scaling
def scale_features(df):
    """Scale numeric features using StandardScaler"""
    df_scaled = df.copy()
    numeric_cols = df_scaled.select_dtypes(include=[np.number]).columns
    
    scaler = StandardScaler()
    df_scaled[numeric_cols] = scaler.fit_transform(df_scaled[numeric_cols])
    
    return df_scaled, scaler

# df_scaled, scaler = scale_features(df)
# print("✅ Features scaled using StandardScaler")

## 7️⃣ GENERATE INSIGHTS REPORT

In [ ]:
def generate_insights_report(profiler):
    """Generate a comprehensive insights report"""
    
    report = {
        'Dataset Overview': {
            'Shape': profiler.df.shape,
            'Memory Usage': f"{profiler.df.memory_usage(deep=True).sum() / 1024**2:.2f} MB",
            'Columns': list(profiler.df.columns)
        },
        'Data Quality': profiler.quality.report,
        'Numeric Features': {
            'Count': len(profiler.numeric.numeric_cols),
            'Features': profiler.numeric.numeric_cols
        },
        'Categorical Features': {
            'Count': len(profiler.categorical.categorical_cols),
            'Features': profiler.categorical.categorical_cols
        },
        'Outliers': profiler.outliers.detect_iqr(),
        'High Correlations': profiler.correlation.get_high_correlations(0.7)
    }
    
    return report

# Generate and print the report
insights = generate_insights_report(profiler)
print("\n\n📊 INSIGHTS REPORT")
print("="*80)
print(json.dumps(insights, indent=2, default=str)[:2000])  # Print first 2000 chars
print("\n... (full report available in 'insights' variable)")

## 8️⃣ SAVE ANALYSIS RESULTS

In [ ]:
# Save insights to JSON
# with open('data_insights.json', 'w') as f:
#     json.dump(insights, f, indent=2, default=str)
# print("✅ Insights saved to 'data_insights.json'")

# Save cleaned data
# df_clean.to_csv('cleaned_data.csv', index=False)
# print("✅ Cleaned data saved to 'cleaned_data.csv'")